# F1 validation (1/2) - EC-Earth3 historical: downscaled vs ERA5 vs raw   [PLAN B]

**Question**: is EC-Earth3 historical downscaled by F1 (analogue method, notebook step1 / step2) closer to ERA5 than the
raw EC-Earth3, and does it look like ERA5 at 0.25 deg (climatology, seasonality, distributions, dry days, extremes,
fine-scale detail, variability)?

**Not day by day**: the analogue method does not synchronise EC-Earth3 and ERA5 days; only **statistics** are compared.

**In-sample or out-of-sample** (`F1_TRAIN_PERIOD`, `EVAL_PERIOD`):
- F1 trained on 1981-2014 and evaluated on 1981-2014: **in-sample**, it only checks that the method does what it should;
- **real validation**: train F1 on 1981-2003 (step1 with `TRAIN_PERIOD = ("1981", "2003")`, then step2), then here
  `F1_TRAIN_PERIOD = ("1981", "2003")` and `EVAL_PERIOD = ("2004", "2014")`.

The raw EC-Earth3 is regridded (conservatively) onto the ERA5 0.25 deg grid for the maps and scores: every field is compared
on the same pixels, inside the domain (`MASK_KIND`). Figures and tables: `outputs/` next to this notebook (not in git).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py, validation_utils.py: this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
import validation_utils as vu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_ALL = cfg["variables"] + cfg.get("extra_variables", [])     # pr, tasmax, tasmin (+ rsds where it exists)

MASK_KIND = "buffer"          # "buffer" = LISFLOOD domain + 0.25 deg buffer | "domain" = without buffer (notebook 00c)
F1_TRAIN_PERIOD = ("1981", "2014")   # training period of the F1 to validate (notebook step1); its outputs: era5like_025/F1_<period>/
OVERWRITE_CACHE = False
DOMAIN = vu.init(cfg, MASK_KIND, borders_res="01M")
DS_DIR = bu.out_dir(cfg, "plan_B", "era5like_025", f"F1_{F1_TRAIN_PERIOD[0]}-{F1_TRAIN_PERIOD[1]}")   # outputs of step2
CACHE = bu.out_dir(cfg, "plan_B", "validation_cache")                 # statistics cache (EOS)
EXPLO_CACHE = bu.out_dir(cfg, "exploration_cache")                    # statistics already computed by 01_exploration
OUT = Path.cwd() / "outputs"                                          # figures + tables, next to this notebook (not in git)
OUT.mkdir(exist_ok=True)
print("downscaled outputs:", DS_DIR, "| figures:", OUT)

EVAL_PERIOD = ("1981", "2014")       # years compared (out-of-sample test: ("2004", "2014") with F1 trained on 1981-2003)
RSDS_PERIOD = ("2005", "2014")       # ERA5 radiation starts in 2005

## 1. Data and statistics (cached)

In [ ]:
def read_downscaled(name, period=None):
    """Downscaled 'ERA5-like' 0.25 deg run written by step2: dict var -> DataArray (standard coordinates)."""
    f = DS_DIR / f"{name}_era5like_025.nc"
    if not f.exists():
        return {}
    ds = xr.open_dataset(f, chunks={"time": 31}, **bu._TIME_KW)
    out = {}
    for v in VARS_ALL:
        if v in ds:
            da = bu.norm_coords(ds[v]).transpose("time", "lat", "lon")
            out[v] = bu.subset_time(da, *period) if period else da
    return out


def read_raw(name, period=None, vars_=None):
    out = {}
    for v in (vars_ or VARS_ALL):
        if bu.dataset_available(cfg, name, v):
            da = bu.read_raw(cfg, name, v)
            out[v] = bu.subset_time(da, *period) if period else da
    return out


def period_of(v):
    return (max(EVAL_PERIOD[0], RSDS_PERIOD[0]), min(EVAL_PERIOD[1], RSDS_PERIOD[1])) if v == "rsds" else EVAL_PERIOD


def per_var(d):
    return {v: bu.subset_time(da, *period_of(v)) for v, da in d.items()}


tag = f"{MASK_KIND}_{EVAL_PERIOD[0]}-{EVAL_PERIOD[1]}"
ds_h = per_var(read_downscaled("gcm_historical"))
assert ds_h, f"No downscaled EC-Earth3 historical in {DS_DIR}: run step1 and step2 first."
VARS = [v for v in VARS_ALL if v in ds_h and bu.dataset_available(cfg, "era5", v) and bu.dataset_available(cfg, "gcm_historical", v)]
print("variables validated:", VARS)

# ERA5 and raw historical: reuse the statistics of 01_exploration when they cover the same period and domain
same_as_explo = EVAL_PERIOD == ("1981", "2014")
def stats_or_explo(name, d, label):
    p_explo = EXPLO_CACHE / f"{name}_stats_{MASK_KIND}.nc"
    if same_as_explo and p_explo.exists() and not OVERWRITE_CACHE:
        print(f"{label}: statistics of 01_exploration reused")
        return xr.open_dataset(p_explo).load()
    return vu.compute_stats(d, CACHE / f"{name}_stats_{tag}.nc", OVERWRITE_CACHE, label=label)

S_era5 = stats_or_explo("era5", per_var(read_raw("era5", vars_=VARS)), "ERA5")
S_raw = stats_or_explo("gcm_historical", per_var(read_raw("gcm_historical", vars_=VARS)), "EC-Earth3 historical raw")
S_ds = vu.compute_stats({v: ds_h[v] for v in VARS}, CACHE / f"gcm_historical_downscaled_F1_{F1_TRAIN_PERIOD[0]}-{F1_TRAIN_PERIOD[1]}_{tag}.nc",
                        OVERWRITE_CACHE, label="EC-Earth3 historical downscaled")
GRID = ds_h[VARS[0]].isel(time=0)                         # ERA5 0.25 deg grid
GCM_GRID = bu.read_raw(cfg, "gcm_historical", VARS[0]).isel(time=0)
S_raw25 = vu.regrid_stats(S_raw, GRID, CACHE / f"gcm_historical_raw_on_era5_grid_{tag}.nc", OVERWRITE_CACHE)
SETS = {"ERA5": S_era5, "raw": S_raw25, "downscaled": S_ds}
COL = {"ERA5": "tab:green", "raw": "k", "downscaled": "tab:red"}
VPLOT = [v for v in ("pr", "tasmax", "tasmin", "dtr", "rsds") if all(vu.has(s, v) for s in SETS.values())]
print("on the ERA5 grid:", bu.same_grid(S_ds[f"{VPLOT[0]}_year_mean"], S_era5[f"{VPLOT[0]}_year_mean"]))

## 2. Integrity of the downscaled output

In [ ]:
rows = []
for v in VARS:
    a = S_ds.attrs
    rows.append({"var": v, "period": a[f"period_{v}"], "n days": a[f"ndays_{v}"], "NaN frac": a[f"nan_frac_{v}"],
                 "min": round(a[f"min_{v}"], 3), "max": round(a[f"max_{v}"], 3),
                 "same grid as ERA5": bu.same_grid(ds_h[v].isel(time=0), S_era5[f"{v}_year_mean"])})
display(pd.DataFrame(rows).set_index("var"))
print("frac pr < -0.01 mm/day:", S_ds.attrs.get("frac_pr_below_m0.01"), "| frac tasmin > tasmax:", S_ds.attrs.get("frac_tasmin_gt_tasmax"))

## 3. Climatologies (annual): ERA5, raw EC-Earth3 (regridded), downscaled

In [ ]:
fig, axes = vu.map_axes(len(VPLOT), 3)
for r, v in enumerate(VPLOT):
    lim = vu.clim_lim(vu.seas(S_era5, v, "ANN"), v)
    if lim is None:
        x = vu.seas(S_era5, v, "ANN").where(vu.mask_for(vu.seas(S_era5, v, "ANN")))
        lim = (float(np.nanpercentile(x, 2)), float(np.nanpercentile(x, 98)))
    for c, (lab, s) in enumerate(SETS.items()):
        vu.draw(axes[r, c], vu.seas(s, v, "ANN"), f"{lab}: {v}", cmap=vu.CMAP[v], lim=lim, label=vu.UNITS[v])
plt.tight_layout(); plt.savefig(OUT / "climatology_era5_raw_downscaled.png", dpi=110); plt.show()

## 4. Bias against ERA5: raw vs downscaled

pr in %. Dots: |Welch t| > 2 on the yearly values (annual column). A good F1 should remove most of the raw bias.

In [ ]:
for v in VPLOT:
    fig, axes = vu.map_axes(2, 3)
    for r, lab in enumerate(("raw", "downscaled")):
        sig = vu.welch_sig(vu.yearly(SETS[lab], v), vu.yearly(S_era5, v))
        lim = None
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            d = vu.diff(vu.seas(SETS[lab], v, s), vu.seas(S_era5, v, s), v)
            if r == 0 and c == 0:
                lim = float(np.nanpercentile(np.abs(d.where(vu.mask_for(d))), 98)) or 1.0
            vu.draw(axes[r, c], d, f"{lab} - ERA5: {v} {s}", cmap=vu.DCMAP[v], sym=True, lim=lim,
                    sig=sig if s == "ANN" else None, label=vu.dlabel(v))
    plt.tight_layout(); plt.savefig(OUT / f"bias_{v}.png", dpi=110); plt.show()

## 5. Scores against ERA5 (over the domain, on the 0.25 deg grid)

`bias` (area-weighted mean difference), `RMSE` (spatial, of the climatology), `corr` (spatial pattern correlation),
`sd ratio` (spatial variability / ERA5). Downscaled should have bias ~ 0, smaller RMSE, corr ~ 1, sd ratio ~ 1.

In [ ]:
rows = []
for v in VPLOT:
    for s in ("ANN", "DJF", "MAM", "JJA", "SON"):
        ref = vu.seas(S_era5, v, s)
        for lab in ("raw", "downscaled"):
            rows.append({"var": v, "season": s, "dataset": lab, **{k: round(x, 3) for k, x in vu.scores(vu.seas(SETS[lab], v, s), ref).items()}})
score_tab = pd.DataFrame(rows).pivot_table(index=["var", "season"], columns="dataset", values=["bias", "RMSE", "corr", "sd ratio"])
score_tab.to_csv(OUT / "scores_vs_era5.csv")
display(score_tab)

## 6. Annual cycle over the domain

In [ ]:
fig, axes = plt.subplots(2, len(VPLOT), figsize=(5 * len(VPLOT), 7), squeeze=False)
for c, v in enumerate(VPLOT):
    cyc = {lab: vu.dmean(s[f"{v}_clim_month"]) for lab, s in SETS.items()}
    for lab, m in cyc.items():
        axes[0, c].plot(m.month, m, "o-", ms=3, color=COL[lab], label=lab)
        if lab != "ERA5":
            axes[1, c].plot(m.month, m - cyc["ERA5"], "s-", ms=3, color=COL[lab], label=f"{lab} - ERA5")
    axes[0, c].set_title(f"{DOMAIN} {v} ({vu.UNITS[v]})"); axes[1, c].axhline(0, color="grey", lw=0.5); axes[1, c].set_xlabel("month")
axes[0, 0].legend(fontsize=7); axes[1, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "annual_cycle.png", dpi=110); plt.show()

## 7. Distributions (pooled over random pixels of the domain, all days)

Q-Q plots against ERA5: the downscaled quantiles should lie on the 1:1 line. (Raw: native pixels; downscaled and ERA5:
0.25 deg pixels, so the raw tails are expected to be weaker.)

In [ ]:
keys = [k for k in ("pr_q_wet", "tasmax_q", "tasmin_q", "dtr_q", "rsds_q") if all(k in s for s in SETS.values())]
fig, axes = plt.subplots(1, len(keys), figsize=(4.5 * len(keys), 4.2), squeeze=False)
for ax, k in zip(axes[0], keys):
    ref = S_era5[k].values
    for lab in ("raw", "downscaled"):
        ax.plot(ref, SETS[lab][k].values, "o-", ms=3, color=COL[lab], label=lab)
    lo, hi = np.nanmin(ref), np.nanmax(ref)
    ax.plot([lo, hi], [lo, hi], color="grey", lw=0.8, ls="--")
    ax.set_xlabel("ERA5 quantile"); ax.set_ylabel("dataset quantile"); ax.set_title(k.replace("_q_wet", " (wet days)").replace("_q", ""))
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "qq_vs_era5.png", dpi=110); plt.show()
qtab = pd.concat({lab: pd.DataFrame({k: s[k].values for k in keys}, index=vu.QUANTILES) for lab, s in SETS.items()}, axis=1)
qtab.to_csv(OUT / "pooled_quantiles.csv"); display(qtab.round(2))
print("pooled dry-day fraction:", {lab: round(s.attrs.get("pooled_dry_frac", np.nan), 3) for lab, s in SETS.items()})

## 8. Dry days and wet-day intensity

In [ ]:
if "pr" in VPLOT:
    fig, axes = vu.map_axes(2, 3)
    for c, (lab, s) in enumerate(SETS.items()):
        vu.draw(axes[0, c], vu.seas(s, "pr", "ANN", "dry"), f"{lab}: dry-day fraction", cmap="YlOrBr", lim=(0, 1))
        vu.draw(axes[1, c], vu.seas(s, "pr", "ANN", "sdii"), f"{lab}: wet-day intensity", cmap="YlGnBu",
                lim=vu.clim_lim(vu.seas(S_era5, "pr", "ANN", "sdii"), "pr"), label="mm/day")
    plt.tight_layout(); plt.savefig(OUT / "dry_days_intensity.png", dpi=110); plt.show()
    rows = []
    for kind, lab in (("dry", "dry-day fraction"), ("sdii", "wet-day intensity")):
        for s in ("raw", "downscaled"):
            rows.append({"index": lab, "dataset": s, **{k: round(x, 3) for k, x in vu.scores(vu.seas(SETS[s], "pr", "ANN", kind), vu.seas(S_era5, "pr", "ANN", kind)).items()}})
    display(pd.DataFrame(rows).set_index(["index", "dataset"]))

## 9. Extremes: Rx1day, TXx, TNn (climatology of the yearly extremes) and bias against ERA5

In [ ]:
evs = [v for v in vu.EXT if all(vu.has(s, v, "year_ext") for s in SETS.values())]
if evs:
    fig, axes = vu.map_axes(3, len(evs))
    rows = []
    for c, v in enumerate(evs):
        name = vu.EXT[v][1]
        e = vu.seas(S_era5, v, "ANN", "ext")
        vu.draw(axes[0, c], e, f"ERA5 {name}", cmap=vu.CMAP[v], lim=vu.clim_lim(e, v), label=vu.UNITS[v])
        lim = None
        for r, lab in enumerate(("raw", "downscaled"), start=1):
            x = vu.seas(SETS[lab], v, "ANN", "ext")
            d = vu.diff(x, e, v)
            lim = lim or float(np.nanpercentile(np.abs(d.where(vu.mask_for(d))), 98)) or 1.0
            vu.draw(axes[r, c], d, f"{lab} - ERA5: {name}", cmap=vu.DCMAP[v], sym=True, lim=lim,
                    sig=vu.welch_sig(SETS[lab][f"{v}_year_ext"], S_era5[f"{v}_year_ext"]), label=vu.dlabel(v))
            rows.append({"index": name, "dataset": lab, **{k: round(x_, 3) for k, x_ in vu.scores(x, e).items()}})
    plt.tight_layout(); plt.savefig(OUT / "extremes.png", dpi=110); plt.show()
    display(pd.DataFrame(rows).set_index(["index", "dataset"]))

## 10. Fine-scale detail added by the downscaling

`detail = climatology - climatology smoothed to the EC-Earth3 resolution` (conservative 0.25 -> 0.7 -> 0.25 deg).
The raw model has (almost) no detail at 0.25 deg; a good downscaling reproduces the ERA5 detail (orography, coasts):
correlation of the downscaled detail with the ERA5 detail close to 1, similar amplitude (sd ratio ~ 1).

In [ ]:
def detail(x):
    coarse = bu.regrid_conservative(x, GCM_GRID.lat.values, GCM_GRID.lon.values)
    return (x - bu.regrid_conservative(coarse, x.lat.values, x.lon.values)).compute()

rows = []
fig, axes = vu.map_axes(len(VPLOT), 3)
EUROPE_VIEW = (-12, 35, 34, 62)       # zoom of the maps only (the scores use the whole domain)
for r, v in enumerate(VPLOT):
    dets = {lab: detail(vu.seas(s, v, "ANN")) for lab, s in SETS.items()}
    lim = float(np.nanpercentile(np.abs(dets["ERA5"].where(vu.mask_for(dets["ERA5"]))), 98)) or 1.0
    for c, lab in enumerate(SETS):
        vu.draw(axes[r, c], dets[lab], f"{lab}: fine-scale detail of {v}", cmap=vu.DCMAP[v], sym=True, lim=lim,
                label=vu.UNITS[v], extent=EUROPE_VIEW)
    for lab in ("raw", "downscaled"):
        sc = vu.scores(dets[lab], dets["ERA5"])
        rows.append({"var": v, "dataset": lab, "corr with ERA5 detail": round(sc["corr"], 3), "sd ratio": round(sc["sd ratio"], 3)})
plt.tight_layout(); plt.savefig(OUT / "fine_scale_detail.png", dpi=110); plt.show()
display(pd.DataFrame(rows).set_index(["var", "dataset"]))

## 11. Interannual variability: std of the yearly means, ratio to ERA5 (1 = same variability)

In [ ]:
fig, axes = vu.map_axes(2, len(VPLOT))
for c, v in enumerate(VPLOT):
    ref = vu.yearly(S_era5, v).std("year")
    for r, lab in enumerate(("raw", "downscaled")):
        vu.draw(axes[r, c], vu.yearly(SETS[lab], v).std("year") / ref, f"{lab} / ERA5: std of yearly {v}", cmap="PuOr_r", lim=(0.5, 1.5))
plt.tight_layout(); plt.savefig(OUT / "interannual_variability_ratio.png", dpi=110); plt.show()

## Conclusions (to fill in)

- Bias removed by F1 (where / which season / which variable): ...
- Distributions, dry days, extremes: ...
- Fine-scale detail: ...
- In-sample or out-of-sample result? (`F1_TRAIN_PERIOD`, `EVAL_PERIOD`): ...